In [11]:
import duckdb
from radar_junior.config import RAW_DIR

In [12]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{RAW_DIR / 'open_jobs_daily.csv'}'").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,open_jobs,BIGINT,YES,None,None,None
2,fresher_friendly_jobs,BIGINT,YES,None,None,None


In [13]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{RAW_DIR/ 'jobs.csv'}'").df()

,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,company,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,url,VARCHAR,YES,None,None,None
4,openings,BIGINT,YES,None,None,None
5,team,VARCHAR,YES,None,None,None
6,category,VARCHAR,YES,None,None,None
7,level,VARCHAR,YES,None,None,None
8,years_experience,BIGINT,YES,None,None,None
9,employment,VARCHAR,YES,None,None,None


In [14]:
duckdb.sql(f"SELECT * FROM '{RAW_DIR / 'jobs.csv'}'LIMIT 5").df()


,source,company,title,url,openings,team,category,level,years_experience,employment,...,skills,pay_min,pay_max,pay_currency,posted,first_seen,closed_at,ghost_score,ghost_reasons,fresher_friendly
0,adp,CITY Laundering,Route Representative,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-07-10,2026-07-13,2026-11-08,<NA>,None,False
1,adp,CITY Laundering,Route Service Apprentice,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,junior,<NA>,full_time,...,None,<NA>,<NA>,None,2026-05-27,2026-05-30,2026-11-08,<NA>,None,True
2,adp,CITY Laundering,Route Service Representative,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-08-25,2026-08-26,2026-11-08,<NA>,None,False
3,adp,CITY Laundering,Textile Control Manager,https://workforcenow.adp.com/mascsr/default/md...,1,None,other,mid,<NA>,NaN,...,None,<NA>,<NA>,None,2026-08-28,2026-08-29,2026-11-08,<NA>,None,False
4,adp,CITY Laundering,Truck Loader/ Receiving Operator,https://workforcenow.adp.com/mascsr/default/md...,1,None,manufacturing,mid,<NA>,full_time,...,None,<NA>,<NA>,None,2026-07-10,2026-07-13,2026-11-08,<NA>,None,False


In [15]:
jobs = duckdb.read_csv(RAW_DIR / "jobs.csv", dtype={"posted": "VARCHAR"})
duckdb.sql("SUMMARIZE SELECT * FROM jobs").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,source,VARCHAR,adp,zoho,84,NaN,NaN,NaN,NaN,NaN,11462654,0.00
1,company,VARCHAR,!K7 Music,🚀YOCHANA,385442,NaN,NaN,NaN,NaN,NaN,11462654,0.00
2,title,VARCHAR,! Berufskraftfahrer im nationalen Fernverkehr ...,🫵🏽SWITCHING CAREER? JOIN US SALES MARKETING RE...,3015815,NaN,NaN,NaN,NaN,NaN,11462654,0.00
3,url,VARCHAR,AMR.group,www.tracygross.com,9610191,NaN,NaN,NaN,NaN,NaN,11462654,7.48
4,openings,BIGINT,1,866,160,1.0385900158898629,0.8827542262413777,1,1,1,11462654,0.00
5,team,VARCHAR,EX-Executive,🧑🏽‍💻 Engineering,100490,NaN,NaN,NaN,NaN,NaN,11462654,32.82
6,category,VARCHAR,administrative,support,22,NaN,NaN,NaN,NaN,NaN,11462654,0.00
7,level,VARCHAR,intern,staff+,7,NaN,NaN,NaN,NaN,NaN,11462654,0.00
8,years_experience,BIGINT,0,20,21,4.5526055524436355,3.3274732239339864,2,4,6,11462654,98.01
9,employment,VARCHAR,contract,part_time,4,NaN,NaN,NaN,NaN,NaN,11462654,29.26


## Conclusiones de `jobs.csv`

**Tamaño:** 11.462.654 ofertas y 23 columnas.

**Columnas que usar con confianza**
- `level` (7 valores, de `intern` a `staff+`) y `category` (22 sectores): sin nulos, sirven para agrupar.
- `fresher_friendly` y `remote`: sin nulos.
- `first_seen` como fecha de publicación, en lugar de `posted`.

**Columnas con muchos huecos**
- `years_experience`: 98 % nulos. La inflación del puesto junior solo se puede medir en ~230.000 ofertas.
- `pay_min` / `pay_max`: 97,7 % nulos. Quedan ~265.000 ofertas con salario, probablemente sesgadas hacia EE. UU.
- `skills`: 85,5 % nulos. Cuando aparece, viene separada por `;`.
- `closed_at`: 85,8 % nulos, pero no es un error: vacío = oferta todavía abierta.

**Problemas de calidad**
- `posted` viene muy sucia (años como `0026`, fechas en otros idiomas). La descarto.
- `pay_max` llega a 315 millones: hay valores extremos que habrá que revisar.
- `pay_currency` mezcla 59 monedas, con mayúsculas y minúsculas inconsistentes.

In [16]:
duckdb.sql("""
    SELECT level, count(*) AS ofertas
    FROM jobs
    GROUP BY level
    ORDER BY ofertas DESC
""")

levelVARCHAR,ofertasBIGINT
mid,10004558
junior,433834
senior,418492
manager+,219528
intern,137895
lead,133788
staff+,114559


In [17]:
duckdb.sql("""
    SELECT level, fresher_friendly, count(*) AS ofertas
    FROM jobs
    GROUP BY level, fresher_friendly
    ORDER BY ofertas DESC
""")

levelVARCHAR,fresher_friendlyBOOLEAN,ofertasBIGINT
mid,False,9975206
junior,True,430804
senior,False,418492
manager+,False,219528
intern,True,137830
lead,False,133788
staff+,False,114559
mid,True,29352
junior,False,3030
intern,False,65


In [18]:
duckdb.sql("""
    SELECT years_experience, count(*) AS ofertas
    FROM jobs
    WHERE level = 'junior' AND fresher_friendly = false
    GROUP BY years_experience
    ORDER BY years_experience
""")

years_experienceBIGINT,ofertasBIGINT
3,1355
4,526
5,539
6,134
7,76
8,93
9,12
10,59
11,3
12,15


In [19]:
duckdb.sql("""
    SELECT
        count(*)                                                AS juniors_con_dato,
        count(*) FILTER (WHERE years_experience >= 3)           AS piden_3_o_mas,
        round(100.0 * piden_3_o_mas / juniors_con_dato, 1)      AS pct_inflacion,

        -- la misma cuenta quitando el pico sospechoso de 18 años
        count(*) FILTER (WHERE years_experience <> 18)          AS juniors_sin_18,
        count(*) FILTER (WHERE years_experience >= 3
                           AND years_experience <> 18)          AS piden_3_o_mas_sin_18,
        round(100.0 * piden_3_o_mas_sin_18 / juniors_sin_18, 1) AS pct_inflacion_sin_18
    FROM jobs
    WHERE level = 'junior'
      AND years_experience IS NOT NULL
""").df()

,juniors_con_dato,piden_3_o_mas,pct_inflacion,juniors_sin_18,piden_3_o_mas_sin_18,pct_inflacion_sin_18
0,38280,3030,7.9,38118,2868,7.5


In [20]:
duckdb.sql("""
    SELECT title, company, category, url
    FROM jobs
    WHERE level = 'junior'
      AND years_experience = 18
    LIMIT 20
""").df()

,title,company,category,url
0,Automotive Sales Associate,Chevroletofwayzata,sales,https://careers.hireology.com/chevroletofwayza...
1,Automotive Sales Associate,Chevroletofwayzata,sales,https://careers.hireology.com/chevroletofwayza...
2,Associate Teacher,Security Benefit,education,https://jobs.dayforcehcm.com/en-US/hcmportal/C...
3,Firebirds Retail Sales Associate - PT,BDA,sales,https://job-boards.greenhouse.io/bdainc/jobs/4...
4,Carpenter Apprentice,ISEC,other,https://jobs.dayforcehcm.com/en-US/isec/CANDID...
5,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/1db23fe5-4ec4-4b9...
6,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/1e4b574c-f925-455...
7,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24059a15-15a9-452...
8,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24abf1e0-84cb-422...
9,Operations Associate,Gopuff,operations,https://jobs.lever.co/gopuff/24c56b8d-d902-4d0...


In [21]:
duckdb.sql("""
    SELECT category, count(*) AS ofertas
    FROM jobs
    WHERE level = 'junior'
      AND years_experience = 18
    GROUP BY category
    ORDER BY ofertas DESC
""").df()

,category,ofertas
0,sales,84
1,operations,37
2,retail,16
3,people,10
4,manufacturing,4
5,logistics,4
6,other,4
7,support,1
8,education,1
9,healthcare,1
